# WC 2026 Knockout Title Probabilities

This notebook computes each still-competing team's probability to win the World Cup.

Workflow:
1. Fetch latest fixtures/results from OpenFootball.
2. Identify unplayed knockout matches and teams with non-zero title chance.
3. Train the hybrid Poisson GLM (same feature pattern as the generation workflow).
4. Compute pairwise knockout advance probabilities.
5. Recursively propagate probabilities through all possible bracket paths.
6. Normalize to 100% and export results.

In [1]:
import json
import urllib.request
import random
import platform
from datetime import datetime
from pathlib import Path
from functools import lru_cache

import numpy as np
import pandas as pd
from scipy.stats import poisson
from sklearn.linear_model import PoissonRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from penaltyblog.ratings import PiRatingSystem

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

ROOT = Path('../../').resolve()
INTERIM = ROOT / 'data' / 'interim'
PROCESSED = ROOT / 'data' / 'processed'
PRED_DIR = ROOT / 'data' / 'predictions'
PRED_DIR.mkdir(parents=True, exist_ok=True)

OPENFOOTBALL_URL = 'https://raw.githubusercontent.com/openfootball/worldcup.json/master/2026/worldcup.json'

NAME_MAP = {
    'United States': 'USA',
    'Bosnia and Herzegovina': 'Bosnia & Herzegovina',
    'Czechia': 'Czech Republic',
    'Democratic Republic of Congo': 'DR Congo',
    'Korea, South': 'South Korea',
    "Cote d'Ivoire": 'Ivory Coast',
    'Bosnia-Herzegovina': 'Bosnia & Herzegovina',
    'Curacao': 'Curaçao',
}

HOME_FEATURES = [
    'home_elo', 'away_elo', 'elo_diff',
    'pi_home', 'pi_away', 'pi_diff',
    'is_neutral',
    'home_form_wr', 'away_form_wr',
    'home_form_gf', 'away_form_gf',
    'home_form_ga', 'away_form_ga',
    'tournament_weight',
]
AWAY_FEATURES = [
    'away_elo', 'home_elo', 'elo_diff',
    'pi_away', 'pi_home', 'pi_diff',
    'is_neutral',
    'away_form_wr', 'home_form_wr',
    'away_form_gf', 'home_form_gf',
    'away_form_ga', 'home_form_ga',
    'tournament_weight',
]
MAX_GOALS = 12

print(f'Python: {platform.python_version()}')
print(f'Numpy: {np.__version__}')
print(f'Pandas: {pd.__version__}')
print(f'Run timestamp (UTC): {datetime.utcnow().isoformat()}')

Python: 3.13.5
Numpy: 2.4.6
Pandas: 2.3.3
Run timestamp (UTC): 2026-06-30T10:05:36.210360


/var/folders/jd/xlgpsgsj3y3dnt7lw6sftwd80000gp/T/ipykernel_28370/372824583.py:63: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  print(f'Run timestamp (UTC): {datetime.utcnow().isoformat()}')


## 1) Fetch latest fixtures and parse knockout structure

In [2]:
def harmonize(name: str | None) -> str | None:
    if name is None:
        return None
    return NAME_MAP.get(name, name)

with urllib.request.urlopen(OPENFOOTBALL_URL, timeout=20) as resp:
    wc_json = json.loads(resp.read())

rows = []
for m in wc_json['matches']:
    score = m.get('score') or {}
    ft = score.get('ft')
    et = score.get('et')
    p = score.get('p')
    rows.append({
        'num': int(m['num']) if m.get('num') is not None else np.nan,
        'date': pd.to_datetime(m['date']),
        'round': m.get('round'),
        'group': m.get('group'),
        'home_team': harmonize(m.get('team1')),
        'away_team': harmonize(m.get('team2')),
        'home_score_ft': np.nan if ft is None else float(ft[0]),
        'away_score_ft': np.nan if ft is None else float(ft[1]),
        'home_score_et': np.nan if et is None else float(et[0]),
        'away_score_et': np.nan if et is None else float(et[1]),
        'home_score_p': np.nan if p is None else float(p[0]),
        'away_score_p': np.nan if p is None else float(p[1]),
        'played': ft is not None,
    })

fixtures_live = pd.DataFrame(rows).sort_values(['date', 'num']).reset_index(drop=True)

# Save a refreshed copy (full bracket, includes winner-slot placeholders).
fixtures_live.to_parquet(INTERIM / 'wc2026_fixtures_live.parquet', index=False)

print(f'Fetched matches: {len(fixtures_live)}')
print(f'Played: {int(fixtures_live.played.sum())} | Unplayed: {int((~fixtures_live.played).sum())}')
print('Saved:', INTERIM / 'wc2026_fixtures_live.parquet')
display(fixtures_live.head(8))

Fetched matches: 104
Played: 76 | Unplayed: 28
Saved: /Users/seppe.vanswegenoven/projects/wk_2026_match_predictor/data/interim/wc2026_fixtures_live.parquet


,num,date,round,group,home_team,away_team,home_score_ft,away_score_ft,home_score_et,away_score_et,home_score_p,away_score_p,played
0,NaN,2026-06-11,Matchday 1,Group A,Mexico,South Africa,2.0,0.0,NaN,NaN,NaN,NaN,True
1,NaN,2026-06-11,Matchday 1,Group A,South Korea,Czech Republic,2.0,1.0,NaN,NaN,NaN,NaN,True
2,NaN,2026-06-12,Matchday 2,Group B,Canada,Bosnia & Herzegovina,1.0,1.0,NaN,NaN,NaN,NaN,True
3,NaN,2026-06-12,Matchday 2,Group D,USA,Paraguay,4.0,1.0,NaN,NaN,NaN,NaN,True
4,NaN,2026-06-13,Matchday 3,Group B,Qatar,Switzerland,1.0,1.0,NaN,NaN,NaN,NaN,True
5,NaN,2026-06-13,Matchday 3,Group C,Brazil,Morocco,1.0,1.0,NaN,NaN,NaN,NaN,True
6,NaN,2026-06-13,Matchday 3,Group C,Haiti,Scotland,0.0,1.0,NaN,NaN,NaN,NaN,True
7,NaN,2026-06-13,Matchday 3,Group D,Australia,Turkey,2.0,0.0,NaN,NaN,NaN,NaN,True


In [3]:
KNOCKOUT_ROUNDS = ['Round of 32', 'Round of 16', 'Quarter-final', 'Semi-final', 'Match for third place', 'Final']
k = fixtures_live[fixtures_live['round'].isin(KNOCKOUT_ROUNDS)].copy()

print('Knockout status:')
for rnd in ['Round of 32', 'Round of 16', 'Quarter-final', 'Semi-final', 'Final']:
    part = k[k['round'] == rnd]
    print(f'  {rnd:<16} played {int(part.played.sum())}/{len(part)}')

pending_knockout = k[~k['played']].sort_values(['date', 'num'])
print('')
print(f'Pending knockout matches: {len(pending_knockout)}')
display(pending_knockout[['num', 'date', 'round', 'home_team', 'away_team']])

Knockout status:
  Round of 32      played 4/16
  Round of 16      played 0/8
  Quarter-final    played 0/4
  Semi-final       played 0/2
  Final            played 0/1

Pending knockout matches: 28


,num,date,round,home_team,away_team
76,77.0,2026-06-30,Round of 32,France,Sweden
77,78.0,2026-06-30,Round of 32,Ivory Coast,Norway
78,79.0,2026-06-30,Round of 32,Mexico,Ecuador
79,80.0,2026-07-01,Round of 32,England,DR Congo
80,81.0,2026-07-01,Round of 32,USA,Bosnia & Herzegovina
81,82.0,2026-07-01,Round of 32,Belgium,Senegal
82,83.0,2026-07-02,Round of 32,Portugal,Croatia
83,84.0,2026-07-02,Round of 32,Spain,Austria
84,85.0,2026-07-02,Round of 32,Switzerland,Algeria
85,86.0,2026-07-03,Round of 32,Argentina,Cape Verde


## 2) Build model and date-aware feature snapshots

In [4]:
train = pd.read_parquet(PROCESSED / 'train.parquet').copy()
matches_hist = pd.read_parquet(INTERIM / 'matches.parquet').copy()
elo_snap = pd.read_parquet(INTERIM / 'elo_wc2026.parquet')[['team', 'rating']].copy()

# Append already-played WC 2026 matches (from live feed) so snapshots are up to date.
played_wc = fixtures_live[fixtures_live['played']].copy()
if len(played_wc):
    wc_results = played_wc[['date', 'home_team', 'away_team', 'home_score_ft', 'away_score_ft']].copy()
    wc_results = wc_results.rename(columns={'home_score_ft': 'home_score', 'away_score_ft': 'away_score'})
    wc_results['tournament'] = 'FIFA World Cup'
    wc_results['city'] = ''
    wc_results['country'] = ''
    wc_results['neutral'] = True
    wc_results['home_score'] = wc_results['home_score'].astype(int)
    wc_results['away_score'] = wc_results['away_score'].astype(int)

    existing_keys = set(zip(matches_hist['date'].astype(str), matches_hist['home_team'], matches_hist['away_team']))
    new_rows = wc_results[~wc_results.apply(lambda r: (str(r['date'])[:10], r['home_team'], r['away_team']) in existing_keys, axis=1)]
    matches_live = pd.concat([matches_hist, new_rows], ignore_index=True).sort_values('date').reset_index(drop=True)
else:
    matches_live = matches_hist.sort_values('date').reset_index(drop=True)

def build_team_view(df: pd.DataFrame) -> pd.DataFrame:
    h = df[['date', 'home_team', 'home_score', 'away_score']].copy()
    h.columns = ['date', 'team', 'goals_scored', 'goals_conceded']
    a = df[['date', 'away_team', 'away_score', 'home_score']].copy()
    a.columns = ['date', 'team', 'goals_scored', 'goals_conceded']
    tv = pd.concat([h, a], ignore_index=True).sort_values(['team', 'date']).reset_index(drop=True)
    tv['win'] = (tv['goals_scored'] > tv['goals_conceded']).astype(float)
    return tv

# Date grid needed for all unresolved knockout matches.
pending_dates = sorted(pending_knockout['date'].unique())

def compute_pi_snapshots(matches_df: pd.DataFrame, dates: list[pd.Timestamp]) -> dict[pd.Timestamp, dict]:
    out = {}
    matches_df = matches_df.sort_values('date').reset_index(drop=True)
    for d in dates:
        pi = PiRatingSystem(alpha=0.15, beta=0.10, k=0.75)
        part = matches_df[matches_df['date'] < d]
        for _, r in part.iterrows():
            pi.update_ratings(r['home_team'], r['away_team'], int(r['home_score'] - r['away_score']))
        out[d] = pi.team_ratings
    return out

def compute_form_snapshots(matches_df: pd.DataFrame, dates: list[pd.Timestamp], n: int = 5) -> dict[pd.Timestamp, dict]:
    out = {}
    tv_full = build_team_view(matches_df)
    for d in dates:
        tv = tv_full[tv_full['date'] < d].copy()
        grp = tv.groupby('team')
        tv['form_wr'] = grp['win'].transform(lambda x: x.rolling(n, min_periods=1).mean())
        tv['form_gf'] = grp['goals_scored'].transform(lambda x: x.rolling(n, min_periods=1).mean())
        tv['form_ga'] = grp['goals_conceded'].transform(lambda x: x.rolling(n, min_periods=1).mean())
        latest = (
            tv.sort_values('date')
            .drop_duplicates(subset='team', keep='last')
            .set_index('team')[['form_wr', 'form_gf', 'form_ga']]
            .to_dict(orient='index')
        )
        out[d] = latest
    return out

pi_snapshots = compute_pi_snapshots(matches_live, pending_dates)
form_snapshots = compute_form_snapshots(matches_live, pending_dates, n=5)

# Training set gets a fixed pre-tournament pi snapshot (same design as generation notebook).
first_pending_date = min(pending_dates)
pre_tourney_pi = pi_snapshots[first_pending_date]
train['pi_home'] = train['home_team'].map(lambda t: pre_tourney_pi.get(t, {}).get('home', 0.0))
train['pi_away'] = train['away_team'].map(lambda t: pre_tourney_pi.get(t, {}).get('away', 0.0))
train['pi_diff'] = train['pi_home'] - train['pi_away']

def build_X(df: pd.DataFrame, features: list[str]) -> np.ndarray:
    return df[features].copy().fillna(df[features].median()).values

X_home = build_X(train, HOME_FEATURES)
X_away = build_X(train, AWAY_FEATURES)
X_tr = np.vstack([X_home, X_away])
y_tr = np.concatenate([train['home_score'].values, train['away_score'].values])

pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('poisson', PoissonRegressor(alpha=0.1, max_iter=300)),
])
pipe.fit(X_tr, y_tr)

print(f'Trained rows: {len(X_tr)}')
print(f'Poisson iterations: {pipe.named_steps["poisson"].n_iter_}')

Trained rows: 22212
Poisson iterations: 15


## 3) Recursive knockout probability engine

In [5]:
matches_by_num = {int(r['num']): r for _, r in fixtures_live.dropna(subset=['num']).iterrows()}

def decided_winner(match_row: pd.Series) -> str | None:
    if not bool(match_row['played']):
        return None
    h_ft, a_ft = match_row['home_score_ft'], match_row['away_score_ft']
    if h_ft > a_ft:
        return str(match_row['home_team'])
    if a_ft > h_ft:
        return str(match_row['away_team'])

    h_et, a_et = match_row['home_score_et'], match_row['away_score_et']
    if pd.notna(h_et) and pd.notna(a_et):
        if h_et > a_et:
            return str(match_row['home_team'])
        if a_et > h_et:
            return str(match_row['away_team'])

    h_p, a_p = match_row['home_score_p'], match_row['away_score_p']
    if pd.notna(h_p) and pd.notna(a_p):
        if h_p > a_p:
            return str(match_row['home_team'])
        if a_p > h_p:
            return str(match_row['away_team'])

    return None

def build_match_features(team_a: str, team_b: str, d: pd.Timestamp) -> pd.DataFrame:
    elo_a = elo_snap.loc[elo_snap['team'] == team_a, 'rating']
    elo_b = elo_snap.loc[elo_snap['team'] == team_b, 'rating']
    home_elo = float(elo_a.iloc[0]) if len(elo_a) else np.nan
    away_elo = float(elo_b.iloc[0]) if len(elo_b) else np.nan

    pi_home = float(pi_snapshots[d].get(team_a, {}).get('home', 0.0))
    pi_away = float(pi_snapshots[d].get(team_b, {}).get('away', 0.0))

    f_home = form_snapshots[d].get(team_a, {})
    f_away = form_snapshots[d].get(team_b, {})

    row = pd.DataFrame([{
        'home_elo': home_elo,
        'away_elo': away_elo,
        'elo_diff': home_elo - away_elo,
        'pi_home': pi_home,
        'pi_away': pi_away,
        'pi_diff': pi_home - pi_away,
        'is_neutral': 1,
        'home_form_wr': f_home.get('form_wr', np.nan),
        'away_form_wr': f_away.get('form_wr', np.nan),
        'home_form_gf': f_home.get('form_gf', np.nan),
        'away_form_gf': f_away.get('form_gf', np.nan),
        'home_form_ga': f_home.get('form_ga', np.nan),
        'away_form_ga': f_away.get('form_ga', np.nan),
        'tournament_weight': 3.0,
    }])
    return row

@lru_cache(maxsize=None)
def pairwise_advance_prob(team_a: str, team_b: str, match_num: int) -> float:
    m = matches_by_num[match_num]
    d = m['date']
    x = build_match_features(team_a, team_b, d)

    lam_home = float(pipe.predict(build_X(x, HOME_FEATURES))[0])
    lam_away = float(pipe.predict(build_X(x, AWAY_FEATURES))[0])

    goals = np.arange(MAX_GOALS + 1)
    ph = poisson.pmf(goals, lam_home)
    pa = poisson.pmf(goals, lam_away)
    joint = np.outer(ph, pa)

    p_draw = float(np.trace(joint))
    p_home_win = float(np.tril(joint, k=-1).sum())

    # Knockout approximation: draw leads to ET/penalties; split equally.
    return p_home_win + 0.5 * p_draw

def slot_to_distribution(slot: str, current_match_num: int) -> dict[str, float]:
    if isinstance(slot, str) and slot.startswith('W') and slot[1:].isdigit():
        prev_num = int(slot[1:])
        return match_winner_distribution(prev_num)
    return {str(slot): 1.0}

@lru_cache(maxsize=None)
def match_winner_distribution(match_num: int) -> dict[str, float]:
    m = matches_by_num[match_num]
    fixed_winner = decided_winner(m)
    if fixed_winner is not None:
        return {fixed_winner: 1.0}

    left = slot_to_distribution(str(m['home_team']), match_num)
    right = slot_to_distribution(str(m['away_team']), match_num)

    out: dict[str, float] = {}
    for t1, p1 in left.items():
        for t2, p2 in right.items():
            path_p = p1 * p2
            if path_p == 0.0:
                continue
            if t1 == t2:
                out[t1] = out.get(t1, 0.0) + path_p
                continue

            p_t1_adv = pairwise_advance_prob(t1, t2, match_num)
            out[t1] = out.get(t1, 0.0) + path_p * p_t1_adv
            out[t2] = out.get(t2, 0.0) + path_p * (1.0 - p_t1_adv)

    return out

final_match_num = int(fixtures_live['num'].max())
title_probs = match_winner_distribution(final_match_num)
raw_sum = float(sum(title_probs.values()))

# Normalize in case of floating error drift.
if raw_sum > 0:
    title_probs = {k: v / raw_sum for k, v in title_probs.items()}

alive_teams = sorted([t for t, p in title_probs.items() if p > 0])
print(f'Teams with non-zero title chance: {len(alive_teams)}')
print(', '.join(alive_teams))
print(f'Probability sum after normalization: {sum(title_probs.values()):.12f}')

Teams with non-zero title chance: 28
Algeria, Argentina, Australia, Austria, Belgium, Bosnia & Herzegovina, Brazil, Canada, Cape Verde, Colombia, Croatia, DR Congo, Ecuador, Egypt, England, France, Ghana, Ivory Coast, Mexico, Morocco, Norway, Paraguay, Portugal, Senegal, Spain, Sweden, Switzerland, USA
Probability sum after normalization: 1.000000000000


In [6]:
res = pd.DataFrame({
    'team': list(title_probs.keys()),
    'title_prob': list(title_probs.values()),
})
res['title_prob_pct'] = 100.0 * res['title_prob']
res = res.sort_values('title_prob', ascending=False).reset_index(drop=True)

print('Top 12 title probabilities:')
display(res.head(12))

total_pct = res['title_prob_pct'].sum()
print(f'Total percentage: {total_pct:.10f}%')
assert abs(total_pct - 100.0) < 1e-6, 'Probabilities do not sum to 100% after normalization.'

Top 12 title probabilities:


,team,title_prob,title_prob_pct
0,Argentina,0.197722,19.772168
1,Spain,0.146872,14.687174
2,France,0.134257,13.425708
3,Brazil,0.105730,10.572992
4,Colombia,0.066008,6.600751
5,England,0.060304,6.030445
6,Portugal,0.057415,5.741548
7,Belgium,0.037458,3.745777
8,Morocco,0.034508,3.450768
9,Paraguay,0.030463,3.046293


Total percentage: 100.0000000000%


In [7]:
today = datetime.utcnow().strftime('%Y%m%d')
out_path = PRED_DIR / f'wc2026_title_probabilities_{today}.csv'
res.to_csv(out_path, index=False)

print(f'Saved: {out_path}')
print(f'Rows: {len(res)}')

Saved: /Users/seppe.vanswegenoven/projects/wk_2026_match_predictor/data/predictions/wc2026_title_probabilities_20260630.csv
Rows: 28


/var/folders/jd/xlgpsgsj3y3dnt7lw6sftwd80000gp/T/ipykernel_28370/4264269613.py:1: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  today = datetime.utcnow().strftime('%Y%m%d')
